<a href="https://colab.research.google.com/github/ankit-rathi/Quantvesting_v3/blob/main/notebooks/01_ONBOARD_MY_PORTFOLIO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Quantvesting | Onboard My Portfolio

### From one portfolio file → first portfolio assessment

This is the **customer-first entry point**. A new user only needs `Symbol`, `Shares` and `AvgCost`. Account labels and investment history are optional.

**Goal:** get a useful portfolio assessment without asking the customer to understand Quantvesting's internal CSV structure.

> **Customer journey:** Minimum-input onboarding → first useful portfolio assessment.


## What the customer provides

Minimum CSV:

```text
Symbol,Shares,AvgCost
TCS,100,3200
INFY,150,1450
HDFCBANK,200,1650
```

Accepted common alternatives include `Ticker`, `Quantity` and `Average Price`. If an account column is absent, holdings are treated as `MAIN`.

Investment history is **optional**. It can be added later for XIRR and transaction-based historical performance.

In [ ]:
!pip install ta -qq

  Preparing metadata (setup.py) ... done


In [ ]:
# 1. Environment
from pathlib import Path
import sys
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
import os

# Point this to the cloned/copied Quantvesting repository.
PROJECT = Path.cwd()
if not (PROJECT / "src" / "quantvesting").exists():
    candidate = Path("/content/drive/My Drive/quantvesting_v3")
    if (candidate / "src" / "quantvesting").exists():
        PROJECT = candidate
    else:
        raise FileNotFoundError(
            "Could not locate the Quantvesting repository. Open this notebook "
            "from the cloned repository or set PROJECT to its root folder."
        )

sys.path.insert(0, str(PROJECT / "src"))

from quantvesting import (
    Quantvesting,
    load_config,
    load_market_data,
    load_portfolio_data,
    onboard_portfolio_csv,
)

print(f"Project: {PROJECT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project: /content/drive/My Drive/quantvesting_v3


In [ ]:
# 2. Customer identity
# For a beta user, use a simple stable identifier.
PORTFOLIO_ID = input("Enter a simple portfolio ID (default: beta_user_001): ").strip() or "beta_user_001"
PORTFOLIO_DIR = PROJECT / "portfolio_data" / PORTFOLIO_ID
MARKET_DATA_DIR = PROJECT / "market_data"

config = load_config(PROJECT / "config" / "strategy.yaml")
qv = Quantvesting(config)
print(f"Portfolio: {PORTFOLIO_ID}")

Enter a simple portfolio ID (default: beta_user_001): vinod
Portfolio: vinod


## 3. Upload the one required file

In Google Colab, the next cell opens a file picker. In local Jupyter, set `INPUT_FILE` to the path of the CSV you received from the customer.

In [ ]:
# 3. Customer portfolio input
INPUT_FILE = None

try:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        INPUT_FILE = next(iter(uploaded.keys()))
except ImportError:
    INPUT_FILE = input("Path to customer's portfolio CSV: ").strip()

if not INPUT_FILE:
    raise ValueError("Please provide the customer's portfolio CSV.")

print(f"Input: {INPUT_FILE}")

Saving myPortfolioStocks.csv to myPortfolioStocks (1).csv
Input: myPortfolioStocks (1).csv


In [ ]:
# 4. Normalize + persist into the existing engine contract
df_portfolio_input, onboarding_report = onboard_portfolio_csv(
    INPUT_FILE,
    PORTFOLIO_DIR,
)

print("ONBOARDING COMPLETE")
print(f"Input rows:      {onboarding_report['input_rows']}")
print(f"Output rows:     {onboarding_report['output_rows']}")
print(f"Unique securities:{onboarding_report['unique_symbols']}")
print(f"Accounts:        {', '.join(onboarding_report['accounts'])}")
print(f"Saved to:        {onboarding_report['output_path']}")

display(df_portfolio_input.head(10))

ONBOARDING COMPLETE
Input rows:      3
Output rows:     3
Unique securities:3
Accounts:        MAIN
Saved to:        /content/drive/My Drive/quantvesting_v3/portfolio_data/vinod/myPortfolioStocks.csv


,Symbol,Shares,AvgCost,InPortfolio
0,TCS,100,3200,MAIN
1,INFY,150,1450,MAIN
2,HDFCBANK,200,1650,MAIN


In [ ]:
# 5. Load shared Quantvesting data + the newly onboarded portfolio
market_data = load_market_data(MARKET_DATA_DIR)
portfolio_data = load_portfolio_data(
    PORTFOLIO_DIR,
    portfolio_id=PORTFOLIO_ID,
)

# Optional: a customer can add myInvestments.csv later.
print("Shared market data loaded:", {k: len(v) for k, v in market_data.items() if hasattr(v, '__len__')})
print("Portfolio holdings:", len(portfolio_data['portfolio_stocks']))

Shared market data loaded: {'prospects': 292, 'screener': 562, 'momentum': 247}
Portfolio holdings: 3


In [ ]:
# 6. First portfolio assessment
df_portfolio, portfolio_summary = qv.portfolio(
    market_data,
    portfolio_data=portfolio_data,
    eod=False,
    portfolio_id=PORTFOLIO_ID,
)

qv.display_run_summary(portfolio_summary)

print("\nPortfolio coverage")
portfolio_symbols = set(df_portfolio["Symbol"].astype(str))
universe_symbols = set(market_data["prospects"]["Symbol"].astype(str))
outside = sorted(portfolio_symbols - universe_symbols)
print(f"Analysed in Quantvesting universe: {len(portfolio_symbols & universe_symbols)}")
print(f"Outside current universe:          {len(outside)}")
if outside:
    print("Outside-universe holdings:", ", ".join(outside[:20]))

## Run date time (IST): 2026-09-04 21:17:15

Deployed:  8.68 L  
Current:  5.42 L  
CAGR/XIRR %: N/A


Portfolio coverage
Analysed in Quantvesting universe: 3
Outside current universe:          0


In [ ]:
# 7. Executive terminal
df_prospects = qv.prospects(
    market_data,
    portfolio_data=portfolio_data,
    include_portfolio=True,
    portfolio_id=PORTFOLIO_ID,
)
df_portfolio_actions = qv.portfolio_actions(df_portfolio)
df_prospect_actions = qv.prospect_actions(df_prospects, top_n=10)
df_rotation = qv.capital_rotation(df_prospects, df_portfolio)

qv.display_terminal(
    df_portfolio=df_portfolio,
    df_prospects=df_prospects,
    portfolio_summary=portfolio_summary,
    df_rotation=df_rotation,
    df_portfolio_actions=df_portfolio_actions,
    df_prospect_actions=df_prospect_actions,
    portfolio_id=PORTFOLIO_ID,
    run_id=df_portfolio.attrs.get("run_id"),
)

{'current': 542320.0,
 'deployed': 867500.0,
 'xirr': nan,
 'target_value': 933471.0,
 'target_profit': 391151.0,
 'target_profit_pct': 72.12549786104145,
 'portfolio_health_pct': 100.0,
 'portfolio_health_value': 542320.0,
 'core_allocation_pct': 100.0,
 'legacy_allocation_pct': 0.0,
 'out_of_universe_allocation_pct': nan,
 'out_of_universe_basis': 'unavailable',
 'top5_concentration_pct': 100.0,
 'top10_concentration_pct': 100.0,
 'top20_concentration_pct': 100.0,
 'pnl': -325180.0,
 'positions': 3,
 'prospects': 60,
 'weighted_remaining_upside_pct': 0.7212549786104145,
 'median_rrr': -0.47,
 'action_count': 0,
 'prospect_candidates': 0,
 'rotation_candidates': 0}

## What becomes available later

| Customer input | Unlocks |
|---|---|
| Portfolio CSV | Portfolio health, allocation, concentration, P/L, FTT/RRR and existing Quantvesting analytics |
| Investment history | XIRR and transaction-based performance |
| EOD runs over time | Historical portfolio/NAV views |

**Important:** the onboarding layer only normalizes customer input. It does not change the Quantvesting investment methodology.

## Next steps

After this first assessment, continue with **02_MY_PORTFOLIO.ipynb** for the ongoing portfolio terminal. Add investment history later if you want transaction-based XIRR and longitudinal analysis.